In [ ]:
# ==================== SummarizationMiddleware 完整实现 ====================

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import ensure_config
from pydantic import BaseModel, Field
from typing import Optional
from dotenv import load_dotenv
import logging

# ==================== 1. 配置日志 ====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
load_dotenv(override=True)

# ==================== 2. 定义工具 ====================
@tool
def search_patent(query: str) -> str:
    """搜索专利数据库"""
    return f"专利搜索结果: 找到与 '{query}' 相关的 3 项专利..."

@tool
def analyze_technology(tech_desc: str) -> str:
    """分析技术可行性"""
    return f"技术分析: '{tech_desc}' 的实现可行性评估完成..."

tools = [search_patent, analyze_technology]

# ==================== 3. 定义上下文 ====================
class UserContext(BaseModel):
    user_id: str = Field(..., description="用户唯一标识")
    department: str = Field(..., description="所属部门")
    max_history_tokens: Optional[int] = Field(default=1000, description="历史消息 token 阈值")

# ==================== 4. 配置中间件 ====================
summarization_middleware = SummarizationMiddleware(
    model=ChatDeepSeek(model="deepseek-chat", temperature=0.1),
    # max_tokens_before_summary=200,          # 历史消息 token 数量超过 200 时触发压缩
    messages_to_keep=5,                     # 保留最近 5 条消息
    summary_prompt="请将以下对话历史进行摘要，保留关键决策点和技术细节：\n\n{messages}\n\n摘要:"   # 摘要提示词
)

# ==================== 5. 创建 Agent ====================
agent = create_agent(
    model=ChatDeepSeek(model="deepseek-chat", temperature=0.2),
    tools=tools,
    middleware=[summarization_middleware],
    context_schema=UserContext,
    debug=True,
)

# ==================== 6. 执行测试 ====================
def run_summarization_test():
    logger.info("开始 SummarizationMiddleware 测试")

    # 创建长对话历史
    long_history = [HumanMessage(content=f"问题 {i+1}: 如何评估某项技术的专利风险？") for i in range(20)]
    logger.info(f"创建了 {len(long_history)} 条消息")

    # 执行
    result = agent.invoke(
        {"messages": long_history},
        context=UserContext(user_id="engineer_001", department="研发部"),
        config=ensure_config({"configurable": {"thread_id": "session_001"}})
    )

    result_messages = result.get("messages", [])
    logger.info(f"执行后消息数: {len(result_messages)}")

    if len(result_messages) < len(long_history):
        logger.info(f"中间件已触发！压缩了 {len(long_history) - len(result_messages)} 条消息")

    return result

# ==================== 7. 运行测试 ====================
result = run_summarization_test()
logger.info("测试完成")

/data/data/com.termux/files/usr/tmp/ipykernel_17911/3570423299.py:39: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  summarization_middleware = SummarizationMiddleware(
2026-05-19 18:58:30,639 - INFO - 开始 SummarizationMiddleware 测试
2026-05-19 18:58:30,641 - INFO - 创建了 20 条消息
2026-05-19 18:58:30,794 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[values] {'messages': [HumanMessage(content='问题 1: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='20fa2351-10c4-4e5b-b5ca-5386b93baef1'), HumanMessage(content='问题 2: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='bacbb717-476e-4ee2-9d09-999e54324f20'), HumanMessage(content='问题 3: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='4213bf39-08eb-4a50-823f-56b12c695141'), HumanMessage(content='问题 4: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='c04df86c-0a80-4fa5-a564-a72dfbbf443d'), HumanMessage(content='问题 5: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='c1d5f4a5-9394-4f01-a49a-1845dbbb6caa'), HumanMessage(content='问题 6: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='af162e5c-9306-4e6c-aa55-5327a48a6d8b'), HumanMessage(content='问题 7: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='1252acc0-a220-42fe-898c-8b912b426cc9'), HumanMessage(content='问题 8: 如何评估某项技术的专利风险？', a

2026-05-19 18:58:31,493 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
2026-05-19 18:58:40,811 - INFO - 执行后消息数: 21
2026-05-19 18:58:40,813 - INFO - 测试完成


[updates] {'model': {'messages': [AIMessage(content='看起来您重复问了同一个问题20次。我来为您系统性地解答：**如何评估某项技术的专利风险？**\n\n这是一个非常重要的问题，以下是完整的评估框架：\n\n---\n\n## 专利风险评估的完整流程\n\n### 一、专利检索与清查（Patent Clearance Search）\n\n1. **确定技术特征**：拆解技术的核心要素、功能模块、实现方法\n2. **构建检索策略**：\n   - 关键词（中英文同义词、近义词）\n   - 专利分类号（IPC、CPC、LOC）\n   - 申请人/发明人\n   - 时间范围\n3. **多数据库检索**：中国专利数据库、USPTO、EPO、WIPO等\n\n### 二、专利筛选与标引\n\n- 阅读专利摘要、权利要求、说明书\n- 标记相关专利（高、中、低相关度）\n- 识别**有效专利**（未过期、维持有效）\n\n### 三、侵权分析（Claim Chart分析）\n\n| 步骤 | 内容 |\n|------|------|\n| 1 | 将技术特征与专利权利要求逐项比对 |\n| 2 | 判断是否落入**字面侵权**（Literal Infringement） |\n| 3 | 判断是否适用**等同原则**（Doctrine of Equivalents） |\n| 4 | 评估是否存在**现有技术抗辩**或**豁免情形** |\n\n### 四、专利有效性分析\n\n- 评估对方专利是否可能被**无效宣告**\n- 检查：新颖性、创造性、实用性、充分公开\n- 检索现有技术以挑战专利有效性\n\n### 五、风险等级评估\n\n| 风险等级 | 描述 | 应对策略 |\n|---------|------|---------|\n| 🔴 高风险 | 明确落入有效专利权利要求范围 | 规避设计、获取许可、无效宣告 |\n| 🟡 中风险 | 可能落入等同范围或专利有效性存疑 | 深入分析、准备抗辩方案 |\n| 🟢 低风险 | 未落入或专利已失效 | 监控更新、定期复查 |\n\n### 六、FTO（自由实施）报告\n\n- 出具正式的**Freedom to Operate**分析报告\n- 记录分析过

2026-05-19 18:58:41,635 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from typing import Callable,TypedDict

# 定义上下文结构
class Context(TypedDict):
    user_role: str  # 用户角色

# 1. 定义两个模型实例
small_model = ChatOpenAI(model="gpt-4o-mini")
large_model = ChatOpenAI(model="gpt-4o")

hard_keywords = ("证明", "推导", "严谨", "规划","复杂", "多步骤", "chain of thought",
                     "step-by-step", "reason step by step", "数学", "逻辑证明", "约束求解")

# 2. 定义动态模型切换的中间件
@wrap_model_call
async def dynamic_model_router(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """
    根据对话上下文动态切换模型
    """
    # 获取当前对话的状态（例如消息列表）
    state = request.state
    print(f"--- [Middleware] 当前对话状态: {state} ---")

    print(f"--- [Middleware] 当前请求对象: {request} ---")
    # context = request.context
    # print(f"--- [Middleware] 当前对话环境上下文: {context} ---")
    # config = request.config
    # print(f"--- [Middleware] 当前对话系统配置: {config} ---")
    messages = state.get("messages", [])
    print(f"--- [Middleware] 当前消息数量: {len(messages)} ---")

    # 获取上下文中的用户角色
    print(f"打印运行时上下文: {request.runtime.context}")

    # === 逻辑判断示例 ===
    # 场景 A: 如果对话轮数超过 5 轮，切换到大模型处理复杂上下文
    if len(messages) > 5:
        print(f"--- [Middleware] 检测到长对话 ({len(messages)} msgs)，切换至 GPT-4o ---")
        # 使用 .override() 方法替换本次调用的模型
        request = request.override(model=large_model)

    # 场景 B: 如果用户输入包含特定关键词 (仅作演示，实际可用分类器)
    elif messages and "复杂分析" in messages[-1].content or any(kw.lower() in messages[-1].content for kw in hard_keywords):
         print("--- [Middleware] 检测到复杂任务，切换至 GPT-4o ---")
         request = request.override(model=large_model)

    else:
        print("--- [Middleware] 使用默认小模型 GPT-4o-mini ---")
        # 默认使用 create_agent 初始化时传入的模型（即 small_model）

    # 继续执行调用
    return await handler(request)

# 3. 定义工具（可选）
@tool
def get_weather(city: str):
    """查询天气"""
    return f"{city} 的天气是晴天"

# 4. 创建 Agent 并注入中间件
agent = create_agent(
    model=small_model,  # 默认模型
    tools=[get_weather],
    middleware=[dynamic_model_router],  # <--- 关键：注入动态路由中间件
    context_schema=Context    # 上下文类型
)

# 5. 测试调用
print(">>> User: 你好")
await agent.ainvoke(
    {"messages": [{"role": "user", "content": "你好"}]},
    context={"user_role": "expert"}
)

print("\n>>> User: 请进行复杂的市场分析（模拟触发切换）")
await agent.ainvoke(
    {"messages": [{"role": "user", "content": "请进行复杂的市场分析"}]},
    context={"user_role": "expert"}
)



>>> User: 你好
--- [Middleware] 当前对话状态: {'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='0daa9d5f-56cc-4386-95e3-22e6fedd2871')]} ---
--- [Middleware] 当前请求对象: ModelRequest(model=ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x6d488e16a0>, async_client=<openai.resources.chat.completions.completions.AsyncCom

{'messages': [HumanMessage(content='请进行复杂的市场分析', additional_kwargs={}, response_metadata={}, id='3b34ef9e-2b69-4ebc-9486-f4d6a6ea4f7e'),
  AIMessage(content='当然可以进行复杂的市场分析，但需要您提供一些具体的信息或要求。例如：\n\n1. 您想分析哪个市场或行业？\n2. 您是否有特定的时间范围或数据要求？\n3. 您是否想要分析某些特定的指标（如市场份额、增长率、竞争对手分析等）？\n4. 您是否有关注的特定区域或国家？\n\n请提供更多细节，以便我为您进行准确和深入的分析。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 46, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 43, 'engine_ttlt_ms': 909, 'pre_inference_ms': 120, 'service_tbt_ms': 8, 'service_ttft_ms': 394, 'service_ttlt_ms': 1249, 'total_duration_ms': 1140, 'user_visible_ttft_ms': 274}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_b0

In [ ]:
# ==================== persist_session 轻量包装器实现 ====================

from langchain.agents.middleware import AgentMiddleware, AgentState
import os
import json
import time
from typing import Dict, Any

# 继承AgentMiddleware 的类也是一个中间件的
class PersistSessionMiddleware(AgentMiddleware):
    def __init__(self, path: str):
        super().__init__()
        self.path = path
        if not os.path.exists(path):
            os.makedirs(path)



    
    def after_model(self, state: AgentState, runtime) -> None:
        """在模型调用后保存会话"""
        # 使用当前时间戳作为简单的会话标识或检查点
        timestamp = int(time.time())
        print(f"打印state以供调试: {state}")
        messages = state.get("messages", [])
        if not messages:
            return

        # 仅保存最后一条 AI 消息作为演示，或者保存整个历史
        filename = os.path.join(self.path, f"state_{timestamp}.json")

        # 简化的序列化
        serialized_msgs = []
        for msg in messages:
            msg_data = {
                "role": msg.type,
                "content": msg.content
            }
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                 msg_data['tool_calls'] = msg.tool_calls
            serialized_msgs.append(msg_data)

        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(serialized_msgs, f, ensure_ascii=False, indent=2)
            print(f" [persist_session] 会话状态已保存: {filename}")
        except Exception as e:
            print(f" [persist_session] 保存失败: {e}")

def persist_session(path: str = "./sessions"):
    """
    persist_session 轻量包装器

    Args:
        path: 会话持久化保存的目录路径
    """
    return PersistSessionMiddleware(path)

# ==================== 使用示例 ====================

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 1. 创建 Agent
# 这里我们使用 persist_session 包装器
agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini"),
    tools=[],
    middleware=[persist_session(path="./my_agent_sessions")]
)

# 2. 运行测试
print(">>> 开始 persist_session 演示...")
# 第一次交互
print("\n>>> User: 这是一个测试消息，请保存。")

# 异步调用
# 在大模型后端的宇宙里，只要开写中间件，async/await 就是全链路至高无上的唯一信仰。
await agent.ainvoke({"messages": [HumanMessage(content="这是一个测试消息，请保存。")]})

>>> 开始 persist_session 演示...

>>> User: 这是一个测试消息，请保存。
打印state以供调试: {'messages': [HumanMessage(content='这是一个测试消息，请保存。', additional_kwargs={}, response_metadata={}, id='3a82deb0-2d3d-427f-b2d8-6db11100caa3'), AIMessage(content='抱歉，我无法保存消息或信息。不过我可以帮助你解答问题或提供信息！请告诉我你需要什么。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 14, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'service_ttft_ms': 781, 'user_visible_ttft_ms': 644, 'total_duration_ms': 1679, 'pre_inference_ms': 137, 'service_tbt_ms': 37, 'service_ttlt_ms': 1805, 'engine_ttft_ms': 84, 'engine_ttlt_ms': 1119, 'engine_tbt_ms': 37}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_4dcfea0a44', 'id': 'chatcmpl-DkLYbHOjZ9oboGy4O5vymIWAfB3a7'

{'messages': [HumanMessage(content='这是一个测试消息，请保存。', additional_kwargs={}, response_metadata={}, id='3a82deb0-2d3d-427f-b2d8-6db11100caa3'),
  AIMessage(content='抱歉，我无法保存消息或信息。不过我可以帮助你解答问题或提供信息！请告诉我你需要什么。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 14, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'service_ttft_ms': 781, 'user_visible_ttft_ms': 644, 'total_duration_ms': 1679, 'pre_inference_ms': 137, 'service_tbt_ms': 37, 'service_ttlt_ms': 1805, 'engine_ttft_ms': 84, 'engine_ttlt_ms': 1119, 'engine_tbt_ms': 37}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_4dcfea0a44', 'id': 'chatcmpl-DkLYbHOjZ9oboGy4O5vymIWAfB3a7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs':